In [26]:
import os
from pathlib import Path
from dipy.io.streamline import load_tractogram
import numpy as np
from hashlib import sha256
import open3d as o3d
import plotly.graph_objects as go
from utils.misc import fps
import torch
from scipy.spatial.distance import cdist


In [27]:
subject = 'sub-1217'
data_path = '/s3/DATA/TractoInferno/derivatives/testset'
dataset = 'TractoInferno'
samples_per_class = 300
samples_per_subject = 10000

In [28]:
def load_stream(trk_path):
    """ Load streamlines from a SINGLE .trk file. """
    bundle_track = load_tractogram(str(trk_path), 'same', bbox_valid_check=False)
    return list(bundle_track.streamlines)
def hash_streamline(streamline):
        return sha256(streamline.tobytes()).hexdigest()
def load_subject(subject_data):
        """ Helper function to load streamlines of all classes for a particular subject. """
        subject, data_path, dataset, sample_per_subject, samples_per_class = subject_data
        subject_id = subject
        streamlines = [] # List of streamlines
        streamline_to_class = {} # Dictionary mapping streamline hash to class name

        if dataset == 'TractoInferno':
            dir_content = Path(os.path.join(data_path, subject, 'tractography')).iterdir()
        elif dataset == '105HCP':
            dir_content = Path(os.path.join(data_path, subject, 'tracts')).iterdir()
        else:
            raise NotImplementedError(f"Dataset {dataset} not implemented.")
        dir_content = sorted([p for p in dir_content if p.is_file() and p.suffix == '.trk'])

        for clss in dir_content:
            if dataset == 'TractoInferno':
                class_name = clss.stem.split('__')[-1]
            elif dataset == '105HCP':
                class_name = clss.stem
            
            # if class_name not in label_to_int_mapping.keys():
            #     continue
            
            if dataset == 'TractoInferno':
                cls_path = os.path.join(data_path, subject, 'tractography', clss)
            elif dataset == '105HCP':
                cls_path = os.path.join(data_path, subject, 'tracts', clss)
            loaded_streamlines = load_stream(cls_path)

            # Optionally downsample if needed
            if len(loaded_streamlines) > samples_per_class:
                loaded_streamlines = [loaded_streamlines[i] for i in np.random.choice(len(loaded_streamlines), samples_per_class, replace=False)]
            
            streamlines.extend(loaded_streamlines)
            streamline_to_class.update({hash_streamline(streamline): class_name for streamline in loaded_streamlines})

        if len(streamlines) > sample_per_subject:
            # streamlines = select_random_set_of_streamlines(streamlines, sample_per_subject)
            streamlines = [streamlines[i] for i in np.random.choice(len(streamlines), sample_per_subject, replace=False)]

        labels = [streamline_to_class[hash_streamline(streamline)] for streamline in streamlines]
        return subject_id, streamlines, labels

def plot_point_cloud(points, red_points=None, title="3D Point Cloud"):
    """
    Plots a 3D point cloud using Plotly.

    Parameters:
        points (numpy.ndarray): A NumPy array of shape (X, 3) containing 3D point coordinates.
        title (str): Title of the plot (default: "3D Point Cloud").
    """
    if not isinstance(points, np.ndarray) or points.shape[1] != 3:
        raise ValueError("Input must be a NumPy array of shape (X, 3)")

    fig = go.Figure()

    fig.add_trace(go.Scatter3d(
        x=points[:, 0],
        y=points[:, 1],
        z=points[:, 2],
        mode='markers',
        marker=dict(size=1, color=points[:, 2], colorscale='Viridis', opacity=0.8)
    ))
    if red_points is not None:
        fig.add_trace(go.Scatter3d(
            x=red_points[:, 0],
            y=red_points[:, 1],
            z=red_points[:, 2],
            mode='markers',
            marker=dict(size=8, color='red', opacity=0.8)
        ))

    fig.update_layout(title=title, scene=dict(
        xaxis_title="X", yaxis_title="Y", zaxis_title="Z"
    ))

    fig.show()

def find_farthest_points_open3d(points, num_points=3):
    """
    Finds `num_points` farthest points in a 3D point cloud using Open3D's KDTree.

    Parameters:
        points (numpy.ndarray): NumPy array of shape (X, 3).
        num_points (int): Number of farthest points to find.

    Returns:
        numpy.ndarray: Farthest `num_points` points (shape: num_points, 3).
    """
    if points.shape[0] < num_points:
        raise ValueError("Not enough points in the point cloud.")

    # Convert NumPy array to Open3D PointCloud
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)

    # Create a KDTree
    kdtree = o3d.geometry.KDTreeFlann(pcd)

    # Start with a random point
    farthest_indices = [np.random.randint(0, len(points))]

    # Iteratively find farthest points
    for _ in range(num_points - 1):
        last_point = points[farthest_indices[-1]]

        # Find the farthest point using KDTree search
        _, idxs, dists = kdtree.search_knn_vector_3d(last_point, len(points))
        next_farthest_index = idxs[np.argmax(dists)]
        
        farthest_indices.append(next_farthest_index)

    return points[farthest_indices]

def find_farthest_points(points, num_points=3):
    """
    Finds `num_points` farthest points in a 3D point cloud.

    Parameters:
        points (numpy.ndarray): NumPy array of shape (X, 3).
        num_points (int): Number of farthest points to find.

    Returns:
        numpy.ndarray: The farthest `num_points` points of shape (num_points, 3).
    """
    if points.shape[0] < num_points:
        raise ValueError("Not enough points in the point cloud.")

    # Compute pairwise distances
    distances = cdist(points, points)

    # Start with a random point
    farthest_indices = [np.random.randint(0, len(points))]

    # Iteratively find farthest points
    for _ in range(num_points - 1):
        # Find the point farthest from the selected set
        min_dist_to_selected = distances[:, farthest_indices].min(axis=1)
        next_farthest_index = np.argmax(min_dist_to_selected)
        farthest_indices.append(next_farthest_index)

    return points[farthest_indices]


In [29]:
subject_id, streamlines, labels = load_subject((subject, data_path, dataset, samples_per_subject, samples_per_class))

In [30]:
mean_pt = np.mean([np.mean(streamline, axis=0) for streamline in streamlines], axis=0)

In [31]:
print(mean_pt.shape)

(3,)


In [32]:
label = ['POPT_L', 'PYT_R']

In [33]:
idxs = [i for i, l in enumerate(labels) if l in label]
streamliness = [streamlines[i] for i in idxs]


In [34]:
points = np.concatenate(streamliness, axis=0)

In [35]:
plot_point_cloud(points, mean_pt[None, :])

In [36]:
label = ['CC_Fr_2', 'CC_Pa','CC_Pr_Po']
idxs = [i for i, l in enumerate(labels) if l in label]
streamliness = [streamlines[i] for i in idxs]
points = np.concatenate(streamliness, axis=0)

In [37]:
plot_point_cloud(points, mean_pt[None,:])

In [38]:
# concatenate all streamlines
# all_streamlines = np.concatenate(streamlines)

In [39]:
# def get_surface(points):
#     pcd = o3d.geometry.PointCloud()
#     pcd.points = o3d.utility.Vector3dVector(points)

#     # Estimate normals
#     pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))

#     # Poisson Surface Reconstruction
#     mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=8)

#     # Sample points from the mesh surface
#     pcd_surface = mesh.sample_points_uniformly(number_of_points=points.shape[0])

#     # Convert back to NumPy array
#     surface_points = np.asarray(pcd_surface.points)
#     return surface_points

In [40]:
# abc = np.random.choice(all_streamlines.shape[0], 10000, replace=False)
# points = all_streamlines[abc]

In [41]:
# fps_3 = find_farthest_points(points, num_points=20)
# fps_3_opne3d = find_farthest_points_open3d(points, num_points=20)

In [42]:
# plot_point_cloud(points, fps_3, title="Farthest Points (FPS) in 3D Point Cloud")

In [43]:
# plot_point_cloud(points, fps_3_opne3d, title="Farthest Points (FPS) in 3D Point Cloud")